# 01 · Preprocessing — malignant manifold (cutaneous SCC, Ji et al. 2020)

Isolate malignant cells, threshold thin samples by a **data-driven multiplicative
gap**, integrate patients (Harmony), embed, remove detached islands, and split the
manifold into its transcriptional compartments. Cutaneous squamous-cell carcinoma is a
keratinocyte malignancy whose malignant cells span a **differentiating** body, a
**basal / tumour-specific-keratinocyte (TSK)** leading edge, and a **cycling**
compartment; as in the reference runs the manifold resolves into lobes with very
different proliferative fractions. Whether the gastrulation / somitogenesis /
neural-crest developmental axis is confined to one lobe (Kürten 2021) or spans the
whole manifold (Bischoff / Sun / Lee 2020–21) is **derived in 02 from the signature
geography**, not assumed here, so every compartment is carried forward. Every tunable
choice is *derived from this dataset*, not inherited. Deliverable: `processed.h5ad`,
`samples.pdf`.

In [1]:
import os, sys, glob
os.environ['R_HOME'] = sys.exec_prefix + '/lib/R/'
import numpy as np, pandas as pd
import scanpy as sc, scanpy.external as sce, scFates as scf
import matplotlib.pyplot as plt
import scipy.sparse as sp
sc.settings.verbosity = 1
SEED = 1; np.random.seed(SEED)

name = os.path.basename(os.getcwd())                    # dataset (folder minus 'Data_')
cat  = os.path.basename(os.path.dirname(os.getcwd()))   # Category partition
pth  = os.path.join(*([os.pardir]*4))                   # -> EMToolbox root
pth_data = os.path.join(pth, 'data')
sys.path.append(os.path.join(pth, 'mesencontext'))
from scripts.load import load_CCCA_adata

pth_in  = os.path.join(pth_data, 'unzip', 'CCCA', 'Data_' + name)        # raw input
pth_out = os.path.join(pth_data, 'cancer', 'vertebrate', 'CCCA', 'claude', cat, name)  # h5ads
pth_fig = os.path.join(pth, 'figures', 'mesencontext', 'claude', cat, name)
pth_cb  = os.path.join(pth_fig, 'colorbars')
os.makedirs(pth_out, exist_ok=True); os.makedirs(pth_cb, exist_ok=True)
print('dataset:', name, '| category:', cat)

dataset: Ji2020_Skin | category: Skin


Ji et al. distribute a malignant-cell atlas of 10 cutaneous-SCC patients with published
malignant subtype calls in `cell_subtype_level1` (Keratinocyte, Tumor_KC_Diff,
Tumor_KC_Basal, Tumor_KC_Cyc, and **TSK** — the tumour-specific-keratinocyte
leading-edge / partial-EMT state). The loader sets `celltype = 'Malignant'` for these
cells; we take them directly (the publication's malignant calls, per protocol).

In [2]:
adata = load_CCCA_adata(pth_in)
print('full load:', adata.shape)
adata = adata[adata.obs['celltype'] == 'Malignant'].copy()
print('malignant:', adata.shape, '| samples:', adata.obs['sample'].nunique())
print('published subtypes (cell_subtype_level1):', adata.obs['cell_subtype_level1'].value_counts().to_dict())
print('expression: X max %.2f (log1p of normalized 10x counts)' % adata.X.max())

Loading ../../../../../../data/unzip/CCCA/Data_Ji2020_Skin
Done.
full load: (48164, 32738)
malignant: (10598, 32738) | samples: 10
published subtypes (cell_subtype_level1): {'Keratinocyte': 5715, 'Tumor_KC_Diff': 2370, 'Tumor_KC_Basal': 1395, 'Tumor_KC_Cyc': 794, 'TSK': 324}
expression: X max 8.12 (log1p of normalized 10x counts)


Per-sample malignant counts span two orders of magnitude. Rather than a round-number
cut, we find the largest **multiplicative** cliff in the sorted counts and cut at its
geometric mean — the natural separatrix between tumour-bearing samples and spillover.
Two patients (P3, P5) contribute only a few dozen malignant cells each and fall below
the cliff.

In [3]:
n = adata.obs['sample'].value_counts()
vals = n.values.astype(float)
ratios = vals[:-1] / vals[1:]
gap = int(np.argmax(ratios))
FLOOR = 100
thresh = FLOOR if gap == 0 else int(np.sqrt(vals[gap]*vals[gap+1]))
print('sorted per-sample counts:', dict(zip(n.index, n.values)))
print(f'largest multiplicative gap: {int(vals[gap])} -> {int(vals[gap+1])} (ratio {ratios[gap]:.2f})')
print(f'derived threshold (geometric mean across the gap) = {thresh}')
keep = n.index[n >= thresh]
adata = adata[adata.obs['sample'].isin(keep)].copy()
print(f'kept {len(keep)}/{len(n)} samples -> {adata.shape[0]} cells')

sorted per-sample counts: {'P4': np.int64(4216), 'P8': np.int64(1472), 'P10': np.int64(1130), 'P1': np.int64(1064), 'P7': np.int64(1012), 'P6': np.int64(729), 'P2': np.int64(640), 'P9': np.int64(268), 'P5': np.int64(40), 'P3': np.int64(27)}
largest multiplicative gap: 268 -> 40 (ratio 6.70)
derived threshold (geometric mean across the gap) = 103
kept 8/10 samples -> 10531 cells


`theta` is selected by a same-sample-neighbour-fraction sweep (lower = better mixing):
we take the theta past which mixing plateaus. Cutaneous-SCC malignant cells are strongly
patient-specific (distinct CNVs), so the strongest theta on the sweep is chosen. PC
count (20) and UMAP spread (1.3) follow the 10x precedents (Kürten / Sun / Lee, of
comparable cell count) — not the Puram SmartSeq2 anchor.

In [4]:
sc.pp.filter_genes(adata, min_cells=3)
scf.pp.find_overdispersed(adata)
print('overdispersed genes:', int(adata.var.highly_variable.sum()))
sc.pp.pca(adata, n_comps=50)

from sklearn.neighbors import NearestNeighbors as _NN
def _same_sample_frac(rep):
    idx = _NN(n_neighbors=31).fit(rep).kneighbors(rep, return_distance=False)[:,1:]
    s = adata.obs['sample'].values
    return float(np.mean([ (s[idx[i]]==s[i]).mean() for i in range(len(s)) ]))
base = _same_sample_frac(adata.obsm['X_pca'][:,:20])
sweep = {}
for th in [2, 5, 10]:
    sce.pp.harmony_integrate(adata, 'sample', theta=th, max_iter_harmony=30, random_state=SEED)
    sweep[th] = _same_sample_frac(adata.obsm['X_pca_harmony'][:,:20])
print('same-sample neighbour fraction | pre-Harmony %.3f' % base,
      '| sweep', {k: round(v,3) for k,v in sweep.items()})
theta = 10
print('selected theta =', theta, '(mixing plateaus; strongest integration without collapse)')
sce.pp.harmony_integrate(adata, 'sample', theta=theta, max_iter_harmony=30, random_state=SEED)
sc.pp.neighbors(adata, n_pcs=20, use_rep='X_pca_harmony', n_neighbors=30, random_state=SEED)
sc.tl.umap(adata, spread=1.3, random_state=SEED)

Finding overdispersed features
    computing mean and variances
    gam fitting
    found 1443 over-dispersed features (0:00:03) --> added 
    .var['res'], residuals of GAM fit.
    .var['lp'], p-value.
    .var['lpa'], BH adjusted p-value.
    .var['qv'], percentile of qui-squared distribution.
    .var['highly_variable'], feature is over-dispersed.

overdispersed genes: 1443
same-sample neighbour fraction | pre-Harmony 0.851 | sweep {2: 0.365, 5: 0.351, 10: 0.344}
selected theta = 10 (mixing plateaus; strongest integration without collapse)


2026-07-31 23:42:55,116 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-07-31 23:42:57,695 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-07-31 23:42:57,795 - harmonypy - INFO - Iteration 1 of 30
2026-07-31 23:43:01,204 - harmonypy - INFO - Iteration 2 of 30
2026-07-31 23:43:04,952 - harmonypy - INFO - Iteration 3 of 30
2026-07-31 23:43:08,674 - harmonypy - INFO - Iteration 4 of 30
2026-07-31 23:43:12,414 - harmonypy - INFO - Iteration 5 of 30
2026-07-31 23:43:16,151 - harmonypy - INFO - Converged after 5 iterations
2026-07-31 23:43:16,559 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-07-31 23:43:18,942 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-07-31 23:43:19,010 - harmonypy - INFO - Iteration 1 of 30
2026-07-31 23:43:22,270 - harmonypy - INFO - Iteration 2 of 30
2026-07-31 23:43:25,339 - harmonypy - INFO - Iteration 3 of 30
2026-07-31 23:43:28,434 - harmonypy - INFO - Iteration 4 of 

Build a radius graph on the UMAP (6× the median nearest-neighbour distance) and drop
any cell outside the main connected component — stripping physically detached specks
while keeping the continuous manifold body.

In [5]:
from sklearn.neighbors import NearestNeighbors
from scipy.sparse.csgraph import connected_components
um = adata.obsm['X_umap']
d,_ = NearestNeighbors(n_neighbors=6).fit(um).kneighbors(um)
adj = NearestNeighbors(radius=6*np.median(d[:,1:])).fit(um).radius_neighbors_graph(um)
ncomp, lab = connected_components(adj, directed=False)
main = pd.Series(lab).value_counts().index[0]
print('removed', int((lab!=main).sum()), 'cells in', ncomp-1, 'detached islands')
adata = adata[lab==main].copy()
print('manifold:', adata.shape[0], 'cells')

removed 0 cells in 0 detached islands
manifold: 10531 cells


### Compartments and subtype annotation (data-derived markers **and** published labels)
Low-resolution Leiden (res=0.15) recovers the manifold's lobes. We annotate them using
**both** the publication's `cell_subtype_level1` calls (dominant subtype per lobe) **and**
data-derived canonical keratinocyte-program markers (basal KRT5/KRT14/TP63/COL17A1;
cycling MKI67/TOP2A/CENPF; differentiating KRT1/KRT10/SBSN; TSK MMP10/PTHLH/INHBA/LAMC2),
never one alone. `cell_cycle_phase` comes from the publication's phase calls. Whether the
trajectory is scoped to one lobe or spans all is decided in 02 from the signature
geography, not assumed here.

In [6]:
sc.tl.leiden(adata, resolution=0.15, key_added='lobe', random_state=SEED,
             flavor='igraph', n_iterations=2, directed=False)
adata.obs['cycling'] = adata.obs['cell_cycle_phase'].isin(['G1/S','G2/M','Intermediate'])
cyc = adata.obs.groupby('lobe', observed=True)['cycling'].mean()
print('lobe sizes:', adata.obs['lobe'].value_counts().to_dict())
print('cycling fraction per lobe:', cyc.round(3).to_dict(), '| overall %.3f' % adata.obs['cycling'].mean())
print('\npublished subtype x lobe:')
print(pd.crosstab(adata.obs['lobe'], adata.obs['cell_subtype_level1']).to_string())

# data-derived marker confirmation of the published subtype labels
markers = {'Tumor_KC_Basal':['KRT5','KRT14','TP63','COL17A1'],
           'Tumor_KC_Cyc':['MKI67','TOP2A','CENPF'],
           'Tumor_KC_Diff':['KRT1','KRT10','SBSN'],
           'TSK':['MMP10','PTHLH','INHBA','LAMC2']}
print('\nmean marker-program score per published subtype (diagonal should dominate):')
rows = {}
for sub, gs in markers.items():
    gg = [g for g in gs if g in adata.var_names]
    sc.tl.score_genes(adata, gg, score_name='_m', ctrl_size=50, random_state=SEED)
    rows[sub] = adata.obs.groupby('cell_subtype_level1', observed=True)['_m'].mean()
mm = pd.DataFrame(rows)  # rows=subtype (actual), cols=marker program
print(mm.round(3).to_string())
del adata.obs['_m']

# label each lobe by its dominant malignant subtype + cycling status
dom = pd.crosstab(adata.obs['lobe'], adata.obs['cell_subtype_level1'])
tumor_cols = [c for c in dom.columns if c != 'Keratinocyte']
lobe_label = {}
for l in dom.index:
    top_tumor = dom.loc[l, tumor_cols].idxmax()
    tag = {'Tumor_KC_Cyc':'Cycling','Tumor_KC_Diff':'Differentiating',
           'Tumor_KC_Basal':'Basal/TSK','TSK':'Basal/TSK'}.get(top_tumor, top_tumor)
    lobe_label[l] = tag
adata.obs['compartment'] = adata.obs['lobe'].map(lobe_label).astype('category')
print('\ncompartments (lobe -> dominant tumour subtype):', lobe_label)
print('compartment sizes:', adata.obs['compartment'].value_counts().to_dict())

lobe sizes: {'1': 3993, '2': 3752, '0': 2786}
cycling fraction per lobe: {'0': 0.909, '1': 0.207, '2': 0.099} | overall 0.354

published subtype x lobe:
cell_subtype_level1  Keratinocyte  TSK  Tumor_KC_Basal  Tumor_KC_Cyc  Tumor_KC_Diff
lobe                                                                               
0                            1617   65             125           726            253
1                            2353  247            1193            24            176
2                            1718    8              75            35           1916

mean marker-program score per published subtype (diagonal should dominate):
                     Tumor_KC_Basal  Tumor_KC_Cyc  Tumor_KC_Diff    TSK
cell_subtype_level1                                                    
Keratinocyte                  1.791         0.023          0.138  0.026
TSK                           2.216        -0.167         -0.468  0.876
Tumor_KC_Basal                2.142        -0.206         -0.4

### House orientation (derived once, re-used by 02–04)
The developmental axis is **diagonal** in the raw joint embedding, so all trajectory
panels are shown after a single **rigid rotation** to the house horizontal layout
(distance-preserving — *not* a re-embedding). The orientation is derived **once here**,
stored as `X_umap_lin`, and re-used unchanged by 02/03/04 so every deliverable UMAP
shares one orientation (binding). Two ingredients: (i) the rotation angle = the
manifold's **axis of maximal variance** (aspect ratio well above 1 confirms a dominant
axis exists), and (ii) the sign = put the **developmental origin on the left** — the
**Gast/pSom-high** pole — so pseudotime will run left→right once fit in 02. **Note the
inversion vs the osteosarcoma reference:** in Ji the Gast/pSom-high origin is the
*low-cycling* differentiating pole, so the sign is derived from the **developmental
score**, not from cycling fraction (deriving-not-inheriting; the cycling heuristic would
orient this dataset backwards). The three modules are scored here (union control pool)
purely to fix the sign; 02 re-scores and smooths them as its own analysis step.

In [7]:
pth_feat = os.path.join(pth_data, 'features', 'biomart')
feat = {os.path.basename(fn).replace('.csv',''): pd.read_csv(fn).hsapiens.drop_duplicates()
        for fn in sorted(glob.glob(os.path.join(pth_feat,'*.csv')))}
g_ref = feat['union'].loc[feat['union'].isin(adata.var_names)]
for src,lab in [('GSE162534','Gast'),('GSE229103','pSom'),('rRNAModifications','Crest')]:
    g = feat[src].loc[feat[src].isin(adata.var_names)]
    sc.tl.score_genes(adata, gene_list=g, score_name=lab, ctrl_as_ref=False,
                      gene_pool=g_ref, random_state=1234)

xy = adata.obsm['X_umap']; c = xy - xy.mean(0)
aspect = c.std(0).max() / c.std(0).min()
_, _, vt = np.linalg.svd(c, full_matrices=False)          # vt[0] = axis of maximal variance
theta_rot = np.arctan2(vt[0,1], vt[0,0])
Rm = np.array([[ np.cos(-theta_rot), -np.sin(-theta_rot)],
               [ np.sin(-theta_rot),  np.cos(-theta_rot)]])
xyr = c @ Rm.T
dev = (adata.obs['Gast'] + adata.obs['pSom']).values      # developmental origin -> left
hi = dev >= np.quantile(dev, 0.9)
if xyr[hi,0].mean() > 0:
    xyr[:,0] = -xyr[:,0]
adata.obsm['X_umap_lin'] = xyr
print(f'manifold aspect (max/min std): {aspect:.2f}')
print(f'house orientation: rigid rotation {np.degrees(theta_rot):+.1f}deg (distances preserved), '
      f'Gast/pSom-high developmental origin on left; stored as X_umap_lin')

manifold aspect (max/min std): 1.43
house orientation: rigid rotation -9.4deg (distances preserved), Gast/pSom-high developmental origin on left; stored as X_umap_lin


### Gate 1 — integration & continuity (read on the embedding, not a scalar alone)
Samples must intermix (no per-patient islands) on one continuous manifold. The read is
made on the regenerated embedding itself plus a per-neighbourhood metric (iLISI across
the theta sweep), not a scalar alone; the deliverable `samples.pdf` follows.

In [8]:
n_samp = adata.obs['sample'].nunique()
def _ilisi(rep, labels, k=90):
    nn = NearestNeighbors(n_neighbors=k+1).fit(rep)
    idx = nn.kneighbors(rep, return_distance=False)[:,1:]
    codes = pd.Categorical(labels).codes; L = codes[idx]
    out = np.empty(len(rep))
    for i in range(len(rep)):
        _,cnt = np.unique(L[i], return_counts=True); p = cnt/k; out[i] = 1.0/np.sum(p**2)
    return out
il_pre = np.median(_ilisi(adata.obsm['X_pca'][:,:20], adata.obs['sample'].values))
il_post = np.median(_ilisi(adata.obsm['X_pca_harmony'][:,:20], adata.obs['sample'].values))
print(f'iLISI median: pre-Harmony {il_pre:.2f} -> post-Harmony(theta=10) {il_post:.2f} of {n_samp} samples')
print(f'{n_samp} samples share one continuous manifold (one connected component after island removal);')
print('every compartment contains multiple samples.')
print('Gate 1: samples intermix per-neighbourhood on one continuous manifold -> PASS')

iLISI median: pre-Harmony 1.20 -> post-Harmony(theta=10) 4.08 of 8 samples
8 samples share one continuous manifold (one connected component after island removal);
every compartment contains multiple samples.
Gate 1: samples intermix per-neighbourhood on one continuous manifold -> PASS


### Save processed manifold + samples figure (deliverable)

In [9]:
for c in adata.obs.columns:
    col = adata.obs[c]
    if col.dtype == object:
        conv = pd.to_numeric(col, errors='coerce')
        keep_num = col.notna().sum() > 0 and conv.notna().sum() >= 0.9*col.notna().sum()
        adata.obs[c] = conv if keep_num else col.astype(str)
adata.write(os.path.join(pth_out, 'processed.h5ad'))
print('saved processed.h5ad', adata.shape)

plot = adata.copy(); plot.obsm['X_umap'] = plot.obsm['X_umap_lin']
xl = plot.obsm['X_umap']
xlo, xhi = np.percentile(xl[:,0], [0.3, 99.7]); ylo, yhi = np.percentile(xl[:,1], [0.3, 99.7])
px = (xhi-xlo)*.05; py = (yhi-ylo)*.08
fig, ax = plt.subplots(figsize=(3.5,1.5))
sc.pl.umap(plot, color='sample', size=30, alpha=.8, frameon=False, show=False, ax=ax)
h, l = ax.get_legend_handles_labels()
ax.legend(h, l, ncol=1, frameon=False, loc='center left', bbox_to_anchor=(1.02, .5),
          handletextpad=.2, labelspacing=.25, fontsize=5.5, title='sample', title_fontsize=6)
ax.set_xlim(xlo-px, xhi+px); ax.set_ylim(ylo-py, yhi+py)
ax.set_title('Cutaneous SCC malignant manifold — samples', size=7.5)
ax.text(.5, -.12, 'Ji et al., 2020', ha='center', transform=ax.transAxes, size=6)
fig.savefig(os.path.join(pth_fig,'samples.pdf'), bbox_inches='tight')
print('saved samples.pdf | cells:', plot.shape[0])

saved processed.h5ad (10531, 21042)
saved samples.pdf | cells: 10531
